# 05. Model Interpretability with SHAP

Explain model predictions using SHAP values.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.config import PROCESSED_DATA_DIR
from src.utils.io_utils import load_parquet
from src.models.cox_model import train_cox_model
from src.evaluation.shap_utils import compute_shap_values, plot_shap_summary, get_top_shap_features

%matplotlib inline

## 1. Load Data and Train Model

In [ ]:
# Load data
train_df = load_parquet(PROCESSED_DATA_DIR / 'train.parquet')
test_df = load_parquet(PROCESSED_DATA_DIR / 'test.parquet')

feature_cols = [c for c in train_df.columns if c not in ['patient_id', 'OS_time', 'OS_status']]

X_train = train_df[feature_cols]
y_train = train_df[['OS_time', 'OS_status']]
X_test = test_df[feature_cols]
y_test = test_df[['OS_time', 'OS_status']]

print(f"Data loaded: {X_train.shape}, {X_test.shape}")

In [ ]:
# Train Cox model
cox_model = train_cox_model(X_train, y_train)
print("Model trained")

## 2. Compute SHAP Values

In [ ]:
# Compute SHAP values (requires SHAP library)
try:
    # Use subset for faster computation
    X_explain = X_test.sample(n=min(100, len(X_test)), random_state=42)
    
    shap_values = compute_shap_values(
        cox_model,
        X_explain,
        model_type='kernel',
        background_samples=50
    )
    
    print(f"SHAP values shape: {shap_values.shape}")
    
except ImportError as e:
    print(f"SHAP not available: {e}")
    print("Install SHAP with: pip install shap")
    shap_values = None

## 3. SHAP Summary Plot

In [ ]:
if shap_values is not None:
    # Plot SHAP summary
    plot_shap_summary(
        shap_values,
        X_explain,
        max_display=20,
        title='SHAP Feature Importance - Cox Model'
    )
else:
    print("Skipping SHAP plots - library not available")

## 4. Top SHAP Features

In [ ]:
if shap_values is not None:
    # Get top features
    top_features = get_top_shap_features(
        shap_values,
        X_explain.columns.tolist(),
        top_n=20
    )
    
    print("Top 20 Features by SHAP:")
    print(top_features)
    
    # Bar plot
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(range(len(top_features)), top_features['mean_abs_shap'])
    ax.set_yticks(range(len(top_features)))
    ax.set_yticklabels(top_features['feature'])
    ax.set_xlabel('Mean |SHAP value|')
    ax.set_title('Top 20 Features by SHAP Importance')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

## 5. Individual Prediction Explanation

In [ ]:
if shap_values is not None:
    from src.evaluation.shap_utils import plot_shap_waterfall
    
    # Explain first patient
    plot_shap_waterfall(
        shap_values,
        X_explain,
        sample_idx=0,
        max_display=10
    )

## Summary

- Computed SHAP values for model explanations
- Identified most important features
- Explained individual predictions

**Note**: SHAP analysis requires the `shap` library. Install with `pip install shap`.

**Limitations**:
- KernelExplainer can be slow for large datasets
- TreeExplainer only works with tree-based models
- SHAP values show feature importance but not causal relationships